# Small-book quote-or-refuse — finetune (2026-10-13)

Before running: **Runtime -> Change runtime type -> A100 GPU**.

Upload only `quote-refuse-2026-10-13.train.jsonl` (126 cards) when asked.
Do not upload the bench file; it is the exam.

If a cell errors, copy the red text back to the agent; library drift happens.

**v2, same day:** trains on the model's answers only (v1 rambled and never
stopped), and has a stop-check cell before export. For a second candidate:
change MODEL_NAME and EXPORT_NAME, Runtime -> Restart session, run all again.

In [ ]:
!pip install -q unsloth

In [ ]:
# Paste the base model your search picked. If unsure, the default below is a safe start.
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
TRAIN_FILE = "quote-refuse-2026-10-13.train.jsonl"
EXPORT_NAME = "smallbook"  # second run: e.g. "smallbook_q3"

from unsloth import FastLanguageModel
import torch

max_seq_length = 4096  # the user message carries 5 passages; 4096 covers them
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose quote-refuse-2026-10-13.train.jsonl

from datasets import load_dataset
dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")

def render(row):
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(render)
print(len(dataset), "cards ready")
print(dataset[0]["text"][:600])

# If this cell errors with "chat template": the base model has none.
# Add before the map:
#   from unsloth import get_chat_template
#   tokenizer = get_chat_template(tokenizer, chat_template="chatml")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=120,  # ~7-8 passes; responses-only training needs more steps
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# v2 fix: train on the model's ANSWERS only — mask the system prompt and
# the passages. v1 trained on the whole text, so it never learned to stop.
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

In [ ]:
# Sanity check BEFORE exporting: does it stop? (v1 rambled forever.)
FastLanguageModel.for_inference(model)
sample = dataset[0]["text"]
prompt_part = sample.split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
inputs = tokenizer([prompt_part], return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=250, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=False)[len(prompt_part):][:700])
# Expect a short reply: [Ch. N] "..." + one gloss line, then it stops itself.
# If it rambles past the reply, tell the agent BEFORE exporting.

In [ ]:
# Export a small file (~2.5 GB) and download it
model.save_pretrained_gguf(EXPORT_NAME, tokenizer, quantization_method="q4_k_m")

import glob, shutil
from google.colab import files

path = sorted(glob.glob("**/*.gguf", recursive=True))[0]
print("gguf:", path)
shutil.make_archive(EXPORT_NAME + "_gguf", "zip", root_dir=path.rsplit("/", 1)[0])
files.download(EXPORT_NAME + "_gguf.zip")